![Databricks Academy](https://files.training.databricks.com/binder/prod_main/devops-essentials-for-data-engineering-en_us-2.2.2/images/20261012T170439Z/DevOps Essentials for Data Engineering/Course Notebooks/Includes/images/icons/databricks_academy.png)

# 11 - Performing Integration Tests

#### Duration: ~15-20 Minutes

Integration tests for data engineering ensure that different components of the data pipeline, such as data ingestion, transformation, storage, and retrieval, work together seamlessly in a real-world environment. These tests validate the flow of data across systems, checking for issues like data consistency, format mismatches, and processing errors when components interact as expected.

There are multiple ways to implement integration tests within Databricks:

1. **Apache Spark Declarative Pipeline (Formerly DLT)**: With Apache Spark Declarative Pipelines, you can use expectations to check pipeline’s results.
    - [Manage data quality with pipeline expectations](https://docs.databricks.com/aws/en/ldp/expectations#manage-data-quality-with-pipeline-expectations)

2. **Jobs**: You can also perform integration tests as a Databricks Jobs with tasks - similarly what is typically done for Non-Spark Declarative Pipeline code.

In this demonstration, we will quickly introduce to you how to perform simple integration tests with Apache Spark Declarative Pipelines and discuss how to implement them with Jobs. Prior knowledge of Apache Spark Declarative Pipeline and Lakeflow Jobs is assumed.

## Learning Objectives

- Learn how to perform integration testing in Apache Spark Declarative Pipelines using expectations.
- Understand how to perform integration tests on data from Spark Declarative Pipeline using Lakeflow Jobs.

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless Compute, Version 5**  
![Serverless Select](https://files.training.databricks.com/binder/prod_main/devops-essentials-for-data-engineering-en_us-2.2.2/images/20261012T170439Z/DevOps Essentials for Data Engineering/Course Notebooks/Includes/images/common/select-serverless.png)
<br></br>
  - How to select an environment version:
[AWS](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version) |
[GCP](https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version)

**NOTE:**  This notebook was **developed and tested using Serverless V5**. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.
  </div>
</div>


## A. Classroom Setup

Run the following cell to configure your working environment for this course. 

##### The notebook "02 - Modularizing PySpark Code - Required" sets up the catalogs for this course. If you have not run this notebook, the catalogs will not be available.

In [0]:
%run ../Includes/Classroom-Setup-Demo

## B. Option 1 - Apache Spark Declarative Pipeline with Integration Tests

In this section, we will create an Apache Spark Declarative Pipeline using the modularized functions from the `src.helpers` file, which we unit tested in the previous notebook. In the pipeline, we will use these functions to create tables and then implement some simple integration tests for the output tables in our ETL pipeline for this project.

- With Apache Spark Declarative Pipelines, you can use expectations to check pipeline’s results.
  - [Manage data quality with pipeline expectations](https://docs.databricks.com/aws/en/ldp/expectations#manage-data-quality-with-pipeline-expectations)

  - [Expectation recommendations and advanced patterns](https://docs.databricks.com/aws/en/ldp/expectation-patterns#expectation-recommendations-and-advanced-patterns)

  - [Applying software development & DevOps best practices to Spark Declarative Pipelines](https://www.databricks.com/blog/applying-software-development-devops-best-practices-delta-live-table-pipelines)


1. We will create the Apache Spark Declarative Pipeline for this project using the Databricks Academy **`DAPipelineConfig`** class, which was specifically designed for this course with the Databricks SDK. This avoids manually creating the pipeline for this demo. Typically during development you would manually build the pipeline with the UI during development.

    **NOTE:** The Databricks SDK is outside the scope of this course. However, if you're interested in seeing the code that uses the SDK to automate building Apache Spark Declarative Pipelines in Databricks Academy, check out the **[../Includes/Classroom-Setup-Common]($../Includes/Classroom-Setup-Common)** notebook in **Cell 6**.

    [Databricks SDK for Python](https://docs.databricks.com/en/dev-tools/sdk-python.html)

    [Databricks SDK Documentation](https://databricks-sdk-py.readthedocs.io/en/latest/)


![Full Lakeflow Spark Declarative Pipeline](https://files.training.databricks.com/binder/prod_main/devops-essentials-for-data-engineering-en_us-2.2.2/images/20261012T170439Z/DevOps Essentials for Data Engineering/Course Notebooks/Includes/images/04_dlt_pipeline.png)

In [0]:
pipeline = DeclarativePipelineCreator(
                            pipeline_name=f"sdk_health_etl_{catalog_dev}", 
                            catalog_name = my_catalog,
                            schema_name = 'dev_ops',
                            root_path_folder_name='src',
                            source_folder_names=[
                                'src/sdp/**', 
                                'tests/integration_test/**'],
                            configuration = {
                                'target': 'development',
                                'raw_data_path':f'/Volumes/{my_catalog}/dev_ops/health'
                            })

pipeline.create_pipeline()

pipeline.start_pipeline()

2. While the Apache Spark Declarative Pipeline is running, examine it through the UI by completing the following steps:

   a. In the far left navigation pane, right-click on **Jobs and Pipelines** and select *Open in a New Tab*.

   b. Find your pipeline named **sdk_health_etl_your_catalog_1_dev** and select it.

   c. Click **Settings** at the top right.

    - c1. In the **Compute** section notice that this pipeline is using **Serverless** compute.

    - c2. On the right side in **Pipeline Settings**, you'll notice that the pipeline contains two **Configuration** variables:

      - **target** = *'development'*
        - This `target` variable will be modified dynamically for each deployment to **development**, **stage**, and **production**.

      - **raw_data_path** = *'/Volumes/your_catalog_1_dev/dev_ops/health'*
        - This `raw_data_path` variable will be modified dynamically for each deployment to **development data**, **stage data**, and **production data**.

    - c3. Click **Cancel** at the bottom right.

   d. At the top of the Pipelines select the kebab menu (three ellipses) and select **View settings YAML**. Notice that the UI provides the necessary YAML files for future deployment. We will talk more about this later. 

   e. In the **Pipeline details** section on the far right, locate the Source code section. You will see a link indicating the number of folders along with an **Open in Editor** option. Click **Open in Editor** to view the notebooks used by the pipeline. Open each notebook in a new tab to examine them:

    - **Notebook 1: [..../src/sdp/ingest-bronze-silver_sdp]($../../src/sdp/ingest-bronze-silver_sdp.py)** - Obtains the Spark Declarative Pipeline configuration variables that setup the target and raw data, and creates the bronze and silver tables based on those variable values.

    - **Notebook 2: [..../src/sdp/gold_tables_sdp]($../../src/sdp/gold_tables_sdp.sql)** - Creates the gold table.

    - **Notebook 3: [..../tests/integration_test/integration_tests_sdp]($../../tests/integration_test/integration_tests_sdp)** - Performs integration tests on the bronze, silver and gold tables based on the target environment. These confirm the row counts, check that the **ID** primary key contains no duplicates, and track out-of-range **Age** and **BMI** values.

   f. Here is a diagram of the entire pipeline for **development, stage and production**. Depending on the values of the **target** and **raw_data_path** configuration variables that are set, the ingest data source and integration tests will vary (dev catalog, stage catalog, prod catalog), but the ETL pipeline will remain the same.

  ![Explain SDP Pipeline](https://files.training.databricks.com/binder/prod_main/devops-essentials-for-data-engineering-en_us-2.2.2/images/20261012T170439Z/DevOps Essentials for Data Engineering/Course Notebooks/Includes/images/04_sdp_explain_integrations.png)

3. **Optional - Profile the silver data to set the range thresholds.**

    The `test_value_ranges` integration test tracks **Age** and **BMI** values that fall outside an expected range. The thresholds used in that test are simple example business rules. After the development pipeline above completes, run the cell below to review the actual **Age** and **BMI** values in the development silver table. You can then adjust the thresholds in **[integration_tests_sdp]($../../tests/integration_test/integration_tests_sdp)** so they fit your data.

**NOTE:** The pipeline you started above runs asynchronously. Run the cell below to wait until the pipeline update finishes before querying its tables (it returns immediately if the pipeline is already complete).

In [0]:
## Wait for the pipeline update to finish before querying the tables it creates
import time
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
for _ in range(60):
    latest = (w.pipelines.get(pipeline.pipeline_id).latest_updates or [None])[0]
    state = latest.state.value if latest and latest.state else None
    print(f"Pipeline update state: {state}")
    if state in ("COMPLETED", "FAILED", "CANCELED"):
        break
    time.sleep(20)

assert state == "COMPLETED", f"The pipeline update ended in state {state}. Open the pipeline UI to review the error."

In [0]:
spark.sql(f"""
    SELECT
        MIN(Age) AS age_min, MAX(Age) AS age_max,
        MIN(BMI) AS bmi_min, MAX(BMI) AS bmi_max
    FROM {my_catalog}.dev_ops.health_silver
""").display()

## C. Option 2 - Integration Testing with Notebooks and Databricks Jobs
You can also perform integration testing using notebooks and add them as tasks in jobs for your pipeline. 

**NOTE:** We will simply review how to implement integration tests with Jobs if that is the method you prefer. The final deployment for this course uses the Apache Spark Declarative Pipeline integration tests with expectations.

#### Steps to take:
1. Create a setup notebook to handle any dynamic setup required using job parameters for your target environment and data locations.

2. Create additional notebooks or files to store the integration tests you want to run as tasks.

3. Organize the new notebooks or files within your **tests** folder.

4. Create a Workflow. Within the Workflow:

   - a. Create the necessary tables or views using Spark Declarative Pipeline or code.

   - b. Add tasks to set up your integration tests (e.g., setting up any dynamic job parameters that need to be set).

   - c. Perform validation by using your notebooks as tasks and set the tasks to all should succeed.

**NOTES:** One major drawback of this approach is that you will need to write more code for setup and validation tasks, as well as manage the job parameters to dynamically modify the code based on the target environment. 

## Summary
Integration testing can be performed in a variety of ways within Databricks. In this demonstration, we focused on how to perform simple integration tests using Spark Declarative Pipeline expectations. We also discussed how to implement them with Workflow tasks.

Depending on your specific situation, you can choose the approach that best fits your needs.


&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>